# PhysX-Anything on Google Colab

Generate simulation-ready 3D assets (URDF/MJCF) from a single iron image.

**Runtime:** Set **Runtime → Change runtime type → GPU** (A100 recommended; T4 may run out of VRAM for the 7B VLM).

Pipeline: `1_vlm_demo.py` → `2_decoder.py` → `3_split.py` → `4_simready_gen.py`

We run it on **two inputs**: an RGB bbox crop (`--remove_bg True`) and an RGBA masked crop (`--remove_bg False`), then compare.

## 1. Check Colab GPU and Environment
Verify the allocated GPU, CUDA version, and Python version.

In [ ]:
!nvidia-smi
!nvcc --version
!python --version
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Clone the Repository with Submodules
PhysX-Anything bundles TRELLIS and other components as submodules, so `--recurse-submodules` is required.

In [ ]:
%cd /content
!git clone --recurse-submodules https://github.com/ziangcao0312/PhysX-Anything.git
%cd /content/PhysX-Anything
!ls

## 3. Install System Dependencies
Colab already has CUDA + PyTorch preinstalled, so we **skip conda** (their `--new-env` flag). We install the system libs needed to build the CUDA rendering extensions (nvdiffrast, diffoctreerast) which require OpenGL/EGL headers.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev \
    libglib2.0-0 libsm6 libxext6 libxrender-dev \
    ninja-build build-essential
print("system deps installed")

## 4. Run the Setup Script (adapted for Colab)

Their `setup.sh --new-env ...` creates a conda env — we DON'T want that on Colab (Colab's Python already has torch+CUDA). Instead we run setup **without** `--new-env`, so it installs into Colab's existing environment.

> ⚠️ This is the **heaviest, riskiest step** (~15-25 min). It builds CUDA extensions (kaolin, nvdiffrast, diffoctreerast, spconv). If a build fails, see the fallback in Section 6.

In [ ]:
# Run their setup WITHOUT --new-env so it uses Colab's Python/torch.
# Note: '. ./setup.sh' style; in Colab we call bash directly.
%cd /content/PhysX-Anything
!bash ./setup.sh --basic --xformers --flash-attn --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast

## 5. Install Qwen2.5 Dependencies
The VLM stage needs specific versions of transformers, qwen-vl-utils, and accelerate.

In [ ]:
!pip install -q transformers==4.50.0
!pip install -q qwen-vl-utils
!pip install -q 'accelerate>=0.26.0'
print("Qwen2.5 deps installed")

## 6. Fallback: Install from requirements.txt
If `setup.sh` (Section 4) failed on some CUDA builds, try installing the bulk deps via `requirements.txt`, then manually add the failed CUDA extensions. Run this **only if Section 4 failed**.

In [ ]:
# FALLBACK ONLY — run if setup.sh failed
%cd /content/PhysX-Anything
!pip install -r requirements.txt

# Manually install the CUDA extensions that setup.sh normally builds:
# kaolin (match torch/cuda version), nvdiffrast, spconv
# !pip install kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
# !pip install git+https://github.com/NVlabs/nvdiffrast.git
# !pip install spconv-cu121

## 7. Verify the Installation
Import the key libraries and print versions to confirm the install succeeded before running inference.

In [ ]:
import importlib

def check(mod):
    try:
        m = importlib.import_module(mod)
        print(f"✅ {mod:15s} {getattr(m, '__version__', '(ok)')}")
    except Exception as e:
        print(f"❌ {mod:15s} FAILED — {e}")

for mod in ["torch", "torchvision", "transformers", "xformers",
            "kaolin", "nvdiffrast", "spconv", "trimesh", "numpy"]:
    check(mod)

import torch
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 8. Download Pretrained Models
Fetch the PhysX-Anything checkpoints (VLM + decoder).

In [ ]:
%cd /content/PhysX-Anything
!python download.py
!ls -la pretrain

## 9. Get the Iron Input Images
Pull the prepared iron crops from your `ego-training` repo (RGB bbox + RGBA masked).

In [ ]:
%cd /content
!git clone https://github.com/abishek21/ego-training.git
%cd /content/PhysX-Anything
!mkdir -p demo_rgb demo_rgba
!cp /content/ego-training/physx_pipeline/iron_f950_rgb.png  demo_rgb/
!cp /content/ego-training/physx_pipeline/iron_f950_rgba.png demo_rgba/
!ls demo_rgb demo_rgba

## 10. Run A — RGB bbox crop (`--remove_bg True`)
Full iron in a box; let PhysX remove the background itself.

In [ ]:
%cd /content/PhysX-Anything
!python 1_vlm_demo.py --demo_path ./demo_rgb --save_part_ply True --remove_bg True --ckpt ./pretrain/vlm
!python 2_decoder.py
!python 3_split.py
!python 4_simready_gen.py --voxel_define 32 --basepath ./test_rgb --process 0 --fixed_base 0 --deformable 0
!echo "=== RGB run outputs ===" && ls -R ./test_rgb | head -40

## 11. Run B — RGBA masked crop (`--remove_bg False`)
Iron already isolated (transparent bg). PhysX skips its own bg removal.

In [ ]:
%cd /content/PhysX-Anything
!python 1_vlm_demo.py --demo_path ./demo_rgba --save_part_ply True --remove_bg False --ckpt ./pretrain/vlm
!python 2_decoder.py
!python 3_split.py
!python 4_simready_gen.py --voxel_define 32 --basepath ./test_rgba --process 0 --fixed_base 0 --deformable 0
!echo "=== RGBA run outputs ===" && ls -R ./test_rgba | head -40

## 12. Download the Results
Zip both runs' outputs (URDF + MJCF + meshes) and download.

In [ ]:
%cd /content/PhysX-Anything
!zip -r -q iron_rgb_asset.zip  ./test_rgb
!zip -r -q iron_rgba_asset.zip ./test_rgba

from google.colab import files
files.download('iron_rgb_asset.zip')
files.download('iron_rgba_asset.zip')

In [ ]:
placeholder